In [3]:
import os
from pathlib import Path

print("Direktori kerja:", Path.cwd())
print("Isi direktori:", os.listdir('/content'))

Direktori kerja: /content
Isi direktori: ['.config', 'demouts', 'sample_data']


In [4]:
from pathlib import Path

PROJECT_DIR = Path('/content/demouts')

folders = [
    PROJECT_DIR / 'notebooks',
    PROJECT_DIR / 'data' / 'raw',
    PROJECT_DIR / 'docs'
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Struktur project berhasil dibuat:")
for path in sorted(PROJECT_DIR.rglob('*')):
    print(path.relative_to(PROJECT_DIR))

Struktur project berhasil dibuat:
data
data/raw
docs
docs/data_ethics_checklist.csv
notebooks


In [5]:
import os
from pathlib import Path

PROJECT_DIR = Path('/content/demouts')

os.chdir(PROJECT_DIR)

print("Direktori kerja:", Path.cwd())

Direktori kerja: /content/demouts


In [6]:
!pip install requests pandas beautifulsoup4

In [7]:
import requests
import pandas as pd
import json
import os
import hashlib
import time

from pathlib import Path
from datetime import datetime
from urllib.robotparser import RobotFileParser

In [8]:
NAMA_PENGAMBIL = " I KADEK ANANTA WIJAYA ASTAWA"
NIM = "2501010124"
EMAIL_KAMPUS = "humas@instiki.ac.id"

HEADERS = {
    "User-Agent": (
        f"demouts/1.0 "
        f"({NAMA_PENGAMBIL}; {EMAIL_KAMPUS})"
    )
}

print(HEADERS)

{'User-Agent': 'demouts/1.0 ( I KADEK ANANTA WIJAYA ASTAWA; humas@instiki.ac.id)'}


In [9]:
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

In [10]:
query = """
[out:json][timeout:90];

area
  ["name"="Bali"]
  ["boundary"="administrative"]
  ["admin_level"="4"]
  ->.bali;

(
  nwr["tourism"~"attraction|waterfall|viewpoint|theme_park|beach|gallery|artwork"](area.bali);
);

out center tags;
"""

In [11]:
import requests
import time

OVERPASS_SERVERS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter"
]

data = None

for server in OVERPASS_SERVERS:

    print("\nMencoba server:", server)

    try:
        response = requests.post(
            server,
            data={"data": query},
            headers=HEADERS,
            timeout=(15, 120)
        )

        print("Status code:", response.status_code)

        response.raise_for_status()

        data = response.json()

        OVERPASS_URL = server

        print("Berhasil mengambil data!")
        print("Jumlah objek:", len(data["elements"]))

        break

    except requests.exceptions.RequestException as e:

        print("Server gagal:", str(e))
        print("Melanjutkan ke server berikutnya...")

    except ValueError as e:

        print("Respons JSON tidak valid:", str(e))

    time.sleep(15)

if data is None:
    raise RuntimeError(
        "Semua server Overpass gagal. "
        "Coba kembali beberapa waktu kemudian."
    )


Mencoba server: https://overpass-api.de/api/interpreter
Status code: 200
Berhasil mengambil data!
Jumlah objek: 921


In [12]:
print(json.dumps(
    data["elements"][:2],
    indent=2,
    ensure_ascii=False
))

[
  {
    "type": "node",
    "id": 253598001,
    "lat": -8.8098072,
    "lon": 115.1682183,
    "tags": {
      "artwork_type": "statue",
      "name": "Vishnu Statue",
      "tourism": "artwork"
    }
  },
  {
    "type": "node",
    "id": 275573942,
    "lat": -8.1757085,
    "lon": 115.1063602,
    "tags": {
      "name": "Small Beautiful Waterfall",
      "tourism": "attraction",
      "waterway": "waterfall"
    }
  }
]


In [13]:
records = []

for item in data["elements"]:

    tags = item.get("tags", {})

    if not tags.get("name"):
        continue

    if item["type"] == "node":
        lat = item.get("lat")
        lon = item.get("lon")
    else:
        center = item.get("center", {})
        lat = center.get("lat")
        lon = center.get("lon")

    records.append({
        "osm_id": item["id"],
        "osm_type": item["type"],
        "nama_tempat": tags.get("name"),
        "kategori_wisata": tags.get("tourism"),
        "kategori_historis": tags.get("historic"),
        "latitude": lat,
        "longitude": lon,
        "website": tags.get("website")
    })

df = pd.DataFrame(records)

print("Jumlah record:", len(df))
print("Jumlah kolom:", len(df.columns))

display(df.head())

Jumlah record: 619
Jumlah kolom: 8


,osm_id,osm_type,nama_tempat,kategori_wisata,kategori_historis,latitude,longitude,website
0,253598001,node,Vishnu Statue,artwork,None,-8.809807,115.168218,None
1,275573942,node,Small Beautiful Waterfall,attraction,None,-8.175709,115.106360,None
2,597770793,node,Sunrise viewpoint of Tanah Lot Temple,viewpoint,None,-8.619558,115.086291,None
3,641637893,node,Bali Butterfly Park,attraction,None,-8.479459,115.134629,None
4,945590562,node,Paintball,attraction,None,-8.817786,115.163878,None


In [14]:
print("Total data:", len(df))
print("Total kolom:", len(df.columns))

if len(df) >= 100 and len(df.columns) >= 5:
    print("Target minimal tugas terpenuhi.")
else:
    print("Target belum terpenuhi. Perluas query atau cakupan data.")

Total data: 619
Total kolom: 8
Target minimal tugas terpenuhi.


In [15]:
from datetime import datetime

tanggal = datetime.now().strftime("%Y%m%d")

output_file = (
    PROJECT_DIR /
    "data" /
    "raw" /
    f"tempat_wisata_bali_{tanggal}.csv"
)

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset disimpan:", output_file)

Dataset disimpan: /content/demouts/data/raw/tempat_wisata_bali_20261116.csv


In [16]:
df_check = pd.read_csv(output_file)

print("Jumlah record:", len(df_check))
print("Jumlah kolom:", len(df_check.columns))

display(df_check.head(10))

Jumlah record: 619
Jumlah kolom: 8


,osm_id,osm_type,nama_tempat,kategori_wisata,kategori_historis,latitude,longitude,website
0,253598001,node,Vishnu Statue,artwork,NaN,-8.809807,115.168218,NaN
1,275573942,node,Small Beautiful Waterfall,attraction,NaN,-8.175709,115.106360,NaN
2,597770793,node,Sunrise viewpoint of Tanah Lot Temple,viewpoint,NaN,-8.619558,115.086291,NaN
3,641637893,node,Bali Butterfly Park,attraction,NaN,-8.479459,115.134629,NaN
4,945590562,node,Paintball,attraction,NaN,-8.817786,115.163878,NaN
5,986320558,node,Jimbaran Beach,beach,NaN,-8.780891,115.163925,NaN
6,989867608,node,Pura Susunan Wadonan,attraction,NaN,-8.740111,115.237100,NaN
7,989874256,node,Patung Catur Muka,artwork,NaN,-8.655969,115.216951,NaN
8,989926064,node,Lipah Beach,beach,NaN,-8.353284,115.689472,NaN
9,1034636100,node,close. private resort,viewpoint,NaN,-8.844187,115.135032,NaN


In [17]:
import hashlib
import json
from datetime import datetime

def hitung_md5(file_path):

    md5 = hashlib.md5()

    with open(file_path, "rb") as file:
        for chunk in iter(
            lambda: file.read(4096),
            b""
        ):
            md5.update(chunk)

    return md5.hexdigest()


provenance = {
    "filename": output_file.name,
    "file_size_bytes": output_file.stat().st_size,
    "md5": hitung_md5(output_file),
    "source": "OpenStreetMap",
    "url": "https://www.openstreetmap.org/",
    "api": OVERPASS_URL,
    "license": "Open Database License (ODbL)",
    "attribution": "© OpenStreetMap contributors",
    "acquisition_time": datetime.now().astimezone().isoformat(),
    "collector": {
        "name": NAMA_PENGAMBIL,
        "nim": NIM
    }
}

provenance_file = PROJECT_DIR / "docs" / "provenance.json"

with open(provenance_file, "w", encoding="utf-8") as file:
    json.dump(
        provenance,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Provenance berhasil dibuat.")

Provenance berhasil dibuat.


demouts/
│
└── docs/
    └── data_ethics_checklist.csv

In [18]:
import pandas as pd
from pathlib import Path


PROJECT_DIR = Path('/content/demouts')

ethics_data = [
    {
        "no": 1,
        "aspek": "Legalitas sumber data",
        "status": "Diperiksa",
        "bukti": "https://www.openstreetmap.org/copyright",
        "keterangan": "Sumber data publik OpenStreetMap dan akses Overpass API diizinkan."
    },
    {
        "no": 2,
        "aspek": "Ketentuan penggunaan API",
        "status": "Diperiksa",
        "bukti": "Endpoint https://overpass-api.de/api/interpreter (Sel 36)",
        "keterangan": "Query dibatasi hanya wilayah Bali untuk kategori wisata sesuai aturan fair use."
    },
    {
        "no": 3,
        "aspek": "Kepatuhan lisensi",
        "status": "Diperiksa",
        "bukti": "docs/provenance.json & bagian 2 README",
        "keterangan": "Lisensi Open Database License (ODbL) dan atribusi © OpenStreetMap contributors telah dicantumkan."
    },
    {
        "no": 4,
        "aspek": "Perlindungan data pribadi",
        "status": "Diperiksa",
        "bukti": "Output tabel df.head() (Sel 38)",
        "keterangan": "Dataset hanya memuat informasi tempat wisata publik, tanpa data pribadi individu."
    },
    {
        "no": 5,
        "aspek": "Transparansi identitas",
        "status": "Diperiksa",
        "bukti": "Variabel HEADERS User-Agent (Sel 11)",
        "keterangan": "User-Agent telah mencantumkan nama pengambil dan email resmi kampus."
    },
    {
        "no": 6,
        "aspek": "Pembatasan frekuensi request",
        "status": "Diperiksa",
        "bukti": "time.sleep(15) dan timeout=(15, 120) (Sel 36)",
        "keterangan": "Diterapkan jeda antar-request selama 15 detik untuk mencegah beban berlebih pada server."
    },
    {
        "no": 7,
        "aspek": "Keamanan kredensial",
        "status": "Diperiksa",
        "bukti": "Seluruh sel kode pada DemoUts.ipynb",
        "keterangan": "Tidak ada API key, token rahasia, atau kata sandi yang tersimpan di dalam notebook."
    },
    {
        "no": 8,
        "aspek": "Integritas data",
        "status": "Diperiksa",
        "bukti": "data/raw/tempat_wisata_bali_20261115.csv & hash MD5 di provenance.json",
        "keterangan": "File dataset mentah tidak diubah secara manual dan memiliki hash MD5 sebagai bukti keaslian."
    },
    {
        "no": 9,
        "aspek": "Dokumentasi provenance",
        "status": "Diperiksa",
        "bukti": "docs/provenance.json & sel dokumentasi README",
        "keterangan": "Sumber, waktu akuisisi, lisensi, dan identitas pengambil telah tercatat lengkap."
    }
]

# Membuat DataFrame
df_ethics = pd.DataFrame(ethics_data)

# Menyimpan CSV
ethics_file = (
    PROJECT_DIR /
    "docs" /
    "data_ethics_checklist.csv"
)

df_ethics.to_csv(
    ethics_file,
    index=False,
    encoding="utf-8-sig"
)

print("Checklist berhasil dibuat!")
print("Lokasi:", ethics_file)
display(df_ethics)

Checklist berhasil dibuat!
Lokasi: /content/demouts/docs/data_ethics_checklist.csv


,no,aspek,status,bukti,keterangan
0,1,Legalitas sumber data,Diperiksa,https://www.openstreetmap.org/copyright,Sumber data publik OpenStreetMap dan akses Ove...
1,2,Ketentuan penggunaan API,Diperiksa,Endpoint https://overpass-api.de/api/interpret...,Query dibatasi hanya wilayah Bali untuk katego...
2,3,Kepatuhan lisensi,Diperiksa,docs/provenance.json & bagian 2 README,Lisensi Open Database License (ODbL) dan atrib...
3,4,Perlindungan data pribadi,Diperiksa,Output tabel df.head() (Sel 38),Dataset hanya memuat informasi tempat wisata p...
4,5,Transparansi identitas,Diperiksa,Variabel HEADERS User-Agent (Sel 11),User-Agent telah mencantumkan nama pengambil d...
5,6,Pembatasan frekuensi request,Diperiksa,"time.sleep(15) dan timeout=(15, 120) (Sel 36)",Diterapkan jeda antar-request selama 15 detik ...
6,7,Keamanan kredensial,Diperiksa,Seluruh sel kode pada DemoUts.ipynb,"Tidak ada API key, token rahasia, atau kata sa..."
7,8,Integritas data,Diperiksa,data/raw/tempat_wisata_bali_20261115.csv & has...,File dataset mentah tidak diubah secara manual...
8,9,Dokumentasi provenance,Diperiksa,docs/provenance.json & sel dokumentasi README,"Sumber, waktu akuisisi, lisensi, dan identitas..."


In [23]:
readme_content = """# Akuisisi Data Tempat Wisata Bali

## 1. Deskripsi Project
Project ini merupakan tugas praktik individu mata kuliah Akuisisi dan Manajemen Data INSTIKI.
Project yang saya buat bertujuan untuk mengambil data pariwisata di Bali menggunakan metode API yang legal.

## 2. Sumber Data
- **Nama:** OpenStreetMap
- **Penerbit:** OpenStreetMap Contributors
- **Website:** https://www.openstreetmap.org/
- **API:** https://overpass-api.de/api/interpreter
- **Lisensi:** Open Database License (ODbL)
- **Atribusi:** © OpenStreetMap contributors.

## 3. Waktu Akuisisi
- **Tanggal dan Waktu:** 2026-11-16 (sesuai catatan pada `docs/provenance.json`)

## 4. Metode Akuisisi
Metode yang digunakan adalah API menggunakan Python `requests`.
- **Endpoint:** https://overpass-api.de/api/interpreter
- **Wilayah pengambilan data:** Bali, Indonesia.
- **Parameter:** Objek dengan kategori `tourism`.

## 5. Etika dan Batasan
- Menggunakan sumber data publik.
- Mengikuti lisensi ODbL.
- Mencantumkan atribusi OpenStreetMap.
- Menggunakan `User-Agent` yang mencantumkan identitas pengambil.
- Menggunakan `timeout` dan jeda antar-request (`time.sleep`).
- Menghindari request berulang yang tidak diperlukan.
- Tidak mengambil data pribadi.
- Tidak menyimpan API key di notebook.

## 6. Penjelasan Kolom Dataset
| Kolom | Keterangan |
| :--- | :--- |
| `osm_id` | ID objek OpenStreetMap |
| `osm_type` | Jenis objek OSM |
| `nama_tempat` | Nama tempat |
| `kategori_wisata` | Kategori wisata |
| `kategori_historis` | Kategori historis |
| `latitude` | Garis lintang |
| `longitude` | Garis bujur |
| `website` | Website tempat jika tersedia |

## 7. Cara Menjalankan
1. Buka Google Colab.
2. Buka notebook di folder `notebooks`.
3. Jalankan seluruh sel dari awal.
4. Pastikan koneksi internet tersedia.
5. Periksa hasil dataset pada `data/raw/`.

## 8. Struktur Project
- `notebooks/`: notebook akuisisi data.
- `data/raw/`: dataset mentah.
- `docs/`: dokumentasi provenance dan etika.
"""

readme_path = PROJECT_DIR / "README.md"
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_content)

print("File README.md berhasil dibuat di:", readme_path)

File README.md berhasil dibuat di: /content/demouts/README.md


In [24]:
for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR))

README.md
data/raw/tempat_wisata_bali_20261116.csv
docs/data_ethics_checklist.csv
docs/provenance.json


In [25]:
import os

print(os.listdir('/content'))

['.config', 'demouts', 'drive', 'demouts.zip', 'sample_data']


In [26]:
!zip -r /content/demouts.zip /content/demouts

updating: content/demouts/ (stored 0%)
updating: content/demouts/docs/ (stored 0%)
updating: content/demouts/docs/data_ethics_checklist.csv (deflated 44%)
updating: content/demouts/docs/provenance.json (deflated 36%)
updating: content/demouts/notebooks/ (stored 0%)
updating: content/demouts/data/ (stored 0%)
updating: content/demouts/data/raw/ (stored 0%)
updating: content/demouts/data/raw/tempat_wisata_bali_20261116.csv (deflated 60%)
  adding: content/demouts/README.md (deflated 51%)


In [27]:
from google.colab import files

files.download('/content/demouts.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>